In [25]:
import pandas as pd
import os

folder = os.path.join('..', 'ASEC_RAW')
file = 'asec2013.dat'
input_file = os.path.join(folder, file)
output_file = 'ASEC_NJ_2013.csv'

if not os.path.exists(input_file):

    print(f"ERROR")

else:
    print(f"{file}")

# Household Variables
h_specs = [
    (0, 1),      # House Bracket
    (1, 6),      # Merge key
    (39, 41),    # State (HG-ST60) (GESTCEN)
    (43,48),     # County (GTCBSA) (2005 and forward)
    (57, 58),    # Sub/Urb (HCCC-R) (GTCBSAST)
    (20, 22),    # Household Size (H-NUMPER) (HRNUMHOU)
    (34, 35),    # Tenure (H-TENURE)
    (247, 255),  # Total Income (HTOTVAL)
    (286, 294),  # Supplemental Weight (HSUP_WGT)
]
h_names = ['Housing_Variable', 'Merge', 'State','County', 'Sub/Urb', 'HH_Size', 'Tenure', 'HH_Income', 'Supp_Weight']

# Person Variables
p_specs = [
    (0, 1),      # Person Bracket
    (1, 6),      # Merge key
    (14, 16),    # Relationship (A-EXPRRP)
    (138, 146),    # Weight (A-FNLWGT)
    (18, 20),    # Age (A-AGE) (PEAGE)
    (24, 26),    # Education (A-HGA) (PEEDUCA)
    (20, 21),    # Marital Status (A-MARITL) (PRMARSTA)
    (579, 587),  # Person Income (PTOTVAL) 
]
p_names = ['Person_Variable', 'Merge', 'Relationship', 'Weight', 'Age', 'Education', 'Marital', 'Person_Income']

households = []
persons = []
nj_house = False

with open(input_file, 'r') as f:
    for line in f:
        
        line = line.replace('\n', '').ljust(500) 
        rectype = line[0]

        if rectype == '1': 
            # Filter for NJ '22'
            if line[39:41] == '22':

                nj_house = True  
                data = [line[s:e].strip() for s, e in h_specs]
                households.append(data)

            else:

                nj_house = False 

        elif rectype == '3':
            # H/P Variable
            if nj_house and line[12:14] in ['01', '02']:

                data = [line[s:e].strip() for s, e in p_specs]
                persons.append(data)

df_h = pd.DataFrame(households, columns=h_names)
df_p = pd.DataFrame(persons, columns=p_names)

nj_df = pd.merge(df_h, df_p, on='Merge', how='inner')

numeric_columns = ['HH_Size', 'Age', 'HH_Income', 'Person_Income']

for col in numeric_columns:

    nj_df[col] = pd.to_numeric(nj_df[col], errors='coerce') 

nj_df['Weight'] = pd.to_numeric(nj_df['Weight'], errors='coerce') / 100

nj_df['Year'] = 1994

nj_df = nj_df.drop(columns=['Housing_Variable', 'Person_Variable'])

nj_df.to_csv(output_file, index=False)


asec2013.dat


In [ ]:
import pandas as pd
import glob

csv_files = glob.glob("*.csv")

df_list = [pd.read_csv(file) for file in csv_files]
merged_df = pd.concat(df_list, ignore_index=True)

merged_df.to_csv("CPS_1993_2013.csv", index=False)
